In [ ]:
import sys
import os

# detect the environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# configure the paths
if IN_COLAB:
    print("Running in Google Colab. Setting up GitHub repo...")
    REPO_URL = "https://github.com/JayC-SF/COMP-432-Project.git"
    REPO_DIR = "/content/COMP-432-Project"
    
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL}
        
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
    
    !pip install optuna
    
    # change the working directory
    os.chdir(REPO_DIR)
else:
    print("Running locally. Setting up relative paths...")
    # move up only if base directory is at notebooks
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
        print(f"Working directory changed to: {os.getcwd()}")
    
    # add working dir to sys path
    if os.getcwd() not in sys.path:
        sys.path.append(os.getcwd())
        
    %load_ext autoreload
    %autoreload 2

from src import preprocess_data as prepd
from src.models.NN import CustomResNet
import src.variables as v
import numpy as np
import torch
from src.train.orchestrator import Orchestrator
from src.utils.hardware import get_device
from src.utils.seed import set_seed
from src.datasets import ICSD_MelSpectogram
from torch.utils.data import DataLoader
import copy
import gc
import optuna
import joblib
set_seed(v.SEED)

Download the mel spectogram `.npz` dataset.

In [ ]:
prepd.download_google_file(v.MEL_SPECTOGRAM_NPZ_FILE_PATH, v.MEL_SPECTOGRAM_NPZ_GID)

In [ ]:
data = np.load(v.MEL_SPECTOGRAM_NPZ_FILE_PATH)
train_ds = ICSD_MelSpectogram(data['X_train'], data['y_train'])
val_ds = ICSD_MelSpectogram(data['X_val'], data['y_val'])
test_ds = ICSD_MelSpectogram(data['X_test'], data['y_test'])

We load the dataset into their respective splits

In [ ]:
set_seed(v.SEED)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

In [ ]:
set_seed(v.SEED)
untrained_custom_resnet_model = CustomResNet(2)

In [ ]:
def create_orchestrator(name, lr, wd, trial=None):
    set_seed(v.SEED)
    
    # use output 2 classes so we can use standard multi class logic for training orchestrator
    LEARNING_RATE = lr
    WEIGHT_DECAY = wd
    PATIENCE = 15
    SAVE_PATH = v.RUNS_PATH/name
    MAX_EPOCHS = 500
    model = copy.deepcopy(untrained_custom_resnet_model)
    DEVICE = get_device()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        weight_decay = WEIGHT_DECAY,
        lr=LEARNING_RATE
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        patience=5,
        factor=0.1,
    )
    criterion = torch.nn.CrossEntropyLoss()

    orchestrator = Orchestrator(
        model=model,
        optimizer=optimizer,
        criterion=criterion,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        patience=PATIENCE,
        save_path=SAVE_PATH,
        scheduler=scheduler,
        max_epochs=MAX_EPOCHS,
        classes=v.CLASSES,
        trial=trial
    )

    return orchestrator

def train_model(name, lr, wd, trial=None):
    orchestrator = create_orchestrator(name, lr, wd, trial)
    print(f"\n🚀 Training {name} | LR: {lr} | WD: {wd}")
    if trial is None:
        print(f"📂 Saving to: {orchestrator.save_path}")

    orchestrator.train()
    return orchestrator




In [ ]:
orchestrator_resnet_baseline = train_model(name="CustomResnet_baseline", lr=5e-4, wd=5e-4)

In [ ]:
results_resnet_baseline = orchestrator_resnet_baseline.test(test_loader)

In [ ]:
def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    wd = trial.suggest_float("wd", 1e-6, 1e-2, log=True)
    
    orchestrator = train_model(name=f"CustomResnet_trial_{trial.number}", lr=lr, wd=wd, trial=trial)
    
    best_val_loss = orchestrator.th.best_val_loss
    
    del orchestrator
    gc.collect() 
    torch.cuda.empty_cache()
    
    return best_val_loss

In [ ]:
study = optuna.create_study(
    direction="minimize", 
    study_name="CustomResnet_ICSD_Optimization"
)

study.optimize(objective, n_trials=15)
# Save the study object to a file
joblib.dump(study, v.RUNS_PATH/"CustomResnet_ICSD_Optimization_Optuna_Study.pkl")

In [ ]:
print("🎯 OPTIMIZATION COMPLETE")
print(f"Best Trial Score: {study.best_value:.4f}")
print(f"Best Hyperparameters: {study.best_params}")

In [ ]:
best_param_orchestrator = train_model(name="CustomResnet_best_params", lr=study.best_params['lr'], wd=study.best_params['wd'])


In [ ]:
best_param_results = best_param_orchestrator.test(test_loader)
